# Main experiment

Run first. Train on unmodified records, calibrate on validation, and compare six detectors on identical simulated faults. Data and generated measurements remain private.


**Data input:** use the original `5EnvSensor_MayToDec2024_180kRows.csv` directly,
or a long-format extract containing CO2, temperature and humidity. Set your path
once in ignored `config/local.json` (copy `config/local.example.json`).
The loader detects the format, selects sensor `6012002000869`, removes identical
duplicates and sorts complete timestamps. It does not alter the source file or
fill missing values. A CO2-only extract is insufficient. See `data/README.md`.


**Code cell 1**

** import the configured local environment**

Dependencies are already installed in this project's .venv; this cell does not download packages.

In [ ]:

# Find the project from its root or the notebooks directory.
from pathlib import Path
import sys
_here=Path.cwd().resolve()
PROJECT_DIR=next((p for p in (_here,*_here.parents) if (p/'src/co2_project').is_dir()),None)
if PROJECT_DIR is None: raise FileNotFoundError('Open this notebook inside the submission project.')
if str(PROJECT_DIR/'src') not in sys.path: sys.path.insert(0,str(PROJECT_DIR/'src'))
from co2_project.project import load_settings, resolve_data_csv, record_run, get_run
from co2_project.data import file_sha256, prepare_environment_csv, make_features, chronological_split, HISTORY_FEATURES
from co2_project.faults import WindowPlanner, write_scenario, inject_fault
from co2_project.models import fit_ols, predict_ols_features, fit_neural, neural_settings
from co2_project.detectors import cusum_series, normalized_cusum, select_cusum_limit
from co2_project.evaluation import evaluate_alarm_events, score_development_events
from functools import partial
SETTINGS=load_settings(PROJECT_DIR)
cusum_series=partial(cusum_series,max_gap_minutes=SETTINGS['MAX_GAP_MINUTES'])
normalized_cusum=partial(normalized_cusum,max_gap_minutes=SETTINGS['MAX_GAP_MINUTES'])
select_cusum_limit=partial(select_cusum_limit,max_gap_minutes=SETTINGS['MAX_GAP_MINUTES'])

import hashlib
import json
import sys
import uuid
import warnings
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from threadpoolctl import threadpool_limits
from IPython.display import display

print('Dependencies ready; no data have been read and no model has been trained.')
from time import perf_counter

import os


**Code cell 2**

** Shared study settings**

Keep baseline settings fixed. Magnitudes are ppm; durations count readings. Rerun from here after changes.

In [ ]:
# Configure raw course data or a full-channel extract through local.json, CO2_DATA_CSV or data/input.csv.
LOCAL_CSV = str(resolve_data_csv(PROJECT_DIR))
TARGET_SENSOR=SETTINGS['TARGET_SENSOR']

FAULT_SEED=SETTINGS['FAULT_SEED']
MODEL_SEED=SETTINGS['MODEL_SEED']
SPIKE_PPM=SETTINGS['SPIKE_PPM']
SPIKE_COUNT=SETTINGS['SPIKE_COUNT']
DRIFT_PPM=SETTINGS['DRIFT_PPM']
DRIFT_READINGS=SETTINGS['DRIFT_READINGS']
DRIFT_COUNT=SETTINGS['DRIFT_COUNT']
BIAS_PPM=SETTINGS['BIAS_PPM']
BIAS_READINGS=SETTINGS['BIAS_READINGS']
BIAS_COUNT=SETTINGS['BIAS_COUNT']

MAX_GAP_MINUTES=SETTINGS['MAX_GAP_MINUTES']
RECOVERY_READINGS=SETTINGS['RECOVERY_READINGS']
THRESHOLD_QUANTILE=SETTINGS['THRESHOLD_QUANTILE']
TREE_MAX_DEPTH=SETTINGS['TREE_MAX_DEPTH']
TREE_MIN_SAMPLES_LEAF=SETTINGS['TREE_MIN_SAMPLES_LEAF']
CUSUM_K_FRACTION=SETTINGS['CUSUM_K_FRACTION']
CUSUM_H_GRID = np.arange(.5,40.5001,.25)  # Candidate h multiples, selected on validation only.

# Create a new directory for each run to preserve earlier results.
base_dir = PROJECT_DIR / 'results/main'
run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")+"_"+uuid.uuid4().hex[:6]
RUN_DIR = base_dir / run_id
print('Private output directory for this run:', RUN_DIR)
print('This cell only sets parameters; models are not trained yet.')
STUDY_VERSION = "nn_threshold_cusum_trial_v1"
for value in (SPIKE_COUNT, DRIFT_COUNT, BIAS_COUNT, DRIFT_READINGS, BIAS_READINGS):
    if type(value) is not int or value < 1:
        raise ValueError('Reading counts, event counts and minimum_rows must be positive integers.')
if not (0 < THRESHOLD_QUANTILE < 1 and RECOVERY_READINGS >= 0 and MAX_GAP_MINUTES > 0
        and all(np.isfinite(value) and value > 0 for value in (SPIKE_PPM, DRIFT_PPM, BIAS_PPM, CUSUM_K_FRACTION))):
    raise ValueError('Check magnitudes, threshold quantile, recovery length and CUSUM settings.')

DEV_SEEDS=SETTINGS['DEV_SEEDS']
DEV_MAGNITUDES=SETTINGS['DEV_MAGNITUDES']
DEV_DURATIONS=SETTINGS['DEV_DURATIONS']
DEV_SPIKE_COUNT=SETTINGS['DEV_SPIKE_COUNT']
DEV_LONG_COUNT=SETTINGS['DEV_LONG_COUNT']
KAPPA_GRID=SETTINGS['KAPPA_GRID']
NORMALIZED_H_GRID = np.arange(0.5, 6.0001, 0.125)
TEST_SEEDS=SETTINGS['TEST_SEEDS']
TEST_MAGNITUDES=SETTINGS['TEST_MAGNITUDES']
TEST_DURATIONS=SETTINGS['TEST_DURATIONS']
RUN_DIR.mkdir(parents=True, exist_ok=False)
experiment_plan = {"selection_data": "faulted validation copies only; test excluded",
    "selection_objective": "equal-type macro paired additional event recall; ties: capped delay, validation alarms, larger H, smaller kappa",
    "development_seeds": DEV_SEEDS, "development_magnitudes": DEV_MAGNITUDES, "development_durations": DEV_DURATIONS,
    "development_event_counts": {"spike": DEV_SPIKE_COUNT, "drift_and_bias": DEV_LONG_COUNT},
    "kappa_grid": KAPPA_GRID, "include_original_normalized_kappa": True,
    "H_grid": NORMALIZED_H_GRID.tolist(), "validation_quantile_budget": THRESHOLD_QUANTILE,
    "test_seeds": TEST_SEEDS, "test_magnitudes": TEST_MAGNITUDES, "test_durations": TEST_DURATIONS,
    "fault_seed": FAULT_SEED, "model_seed": MODEL_SEED,
    "default_faults": {"spike_ppm": SPIKE_PPM,"spike_count": SPIKE_COUNT,"drift_ppm":DRIFT_PPM,"drift_readings":DRIFT_READINGS,"drift_count":DRIFT_COUNT,
                       "bias_ppm":BIAS_PPM,"bias_readings":BIAS_READINGS,"bias_count":BIAS_COUNT}}
(RUN_DIR/"experiment_plan.json").write_text(json.dumps(experiment_plan,indent=2),encoding="utf-8")


**Code cell 3**

**Read the local CSV**

Read the configured file directly. No browser upload or Colab account is used.

In [ ]:
INPUT_CSV = Path(LOCAL_CSV)
if not INPUT_CSV.is_file():
    raise FileNotFoundError(INPUT_CSV)


SOURCE_HASH = file_sha256(INPUT_CSV)
print('Input file:', INPUT_CSV.name)


**Code cell 4**

** Validate and prepare data**

Validate the long table and produce one row per timestamp for CO2, temperature and humidity. Reject invalid values/conflicts; do not interpolate. Pressure is not used in this five-method study.

In [ ]:

original, audit = prepare_environment_csv(INPUT_CSV, TARGET_SENSOR)
print('Detected input format:', audit['input_format'])
print('Complete distinct timestamps:', len(original))
print('Identical channel duplicates removed:', audit['identical_duplicates_removed'])
print('Prepared data SHA-256:', audit['prepared_sha256'])
if audit['input_format']=='raw_course_csv':
    print('Raw rows read:', audit['source_rows_read'], '| selected sensor rows:', audit['selected_sensor_rows'],
          '| invalid JSON rows:', audit['invalid_json_rows'])
display(original.head(3))
print('This cell prepares data only; no training has occurred.')


**Code cell 5**

** Build features and chronological splits**

Create split indices without training. All three fitted predictors share eight inputs; time features retain UTC encoding. Lag6 means six prior readings.

The sensor identifier is a device ID, not a CO2 reading. The location table labels 6012002000869 as Building 60 / G.38 / Compactus_upper.

| Input | Rationale and limitation |
|---|---|
| CO2 lag1 and lag6 | Recent and earlier states; faults can corrupt both |
| Current temperature and humidity | Environmental context; neither uniquely determines true CO2 |
| Hour sin/cos and weekend | Potential repeated time patterns; currently encoded in UTC |
| Minutes since previous reading | Sampling-interval context; exclude history windows with gaps above 20 minutes |

These are selection reasons, not proof that every feature improves detection. Counts such as 7,373 describe rows; no model has been fitted yet.

In [ ]:

# The six-input trial keeps exactly the same eligibility mask as the main study.
prepared=chronological_split(original,MAX_GAP_MINUTES)
original_features=prepared['features']
ELIGIBLE_IDX=prepared['eligible']
TRAIN_IDX,VALIDATION_IDX,TEST_IDX=prepared['train'],prepared['validation'],prepared['test']
split=prepared['split']
MONITORED_IDX=VALIDATION_IDX.append(TEST_IDX)
train_y=original.loc[TRAIN_IDX,'Carbon dioxide']
calibration_observations=original.loc[:VALIDATION_IDX[-1]].copy()
print(f'Train {len(TRAIN_IDX):,}; validation {len(VALIDATION_IDX):,}; test {len(TEST_IDX):,}; excluded {(split=="excluded").sum():,}')

print('Shared inputs:',HISTORY_FEATURES)


**Code cell 6**

Window tools: test by default, validation copies for development. Training injection is rejected.

In [ ]:

planner=WindowPlanner(original,split,MAX_GAP_MINUTES,RECOVERY_READINGS)
plan_windows=planner.plan
save_scenario=partial(write_scenario,original=original,split=split,run_dir=RUN_DIR,
                      fault_seed=FAULT_SEED,recovery_readings=RECOVERY_READINGS,source_hash=SOURCE_HASH)
scenarios={}
print('Fault planner ready. No faults have been created.')


**Code cell 7**

** Shared regression, residual and calibration helpers**

Residual = observed minus predicted. Fit on training only; calibrate point thresholds on unmodified validation only. Define helpers first.

The point residual is $e_t=y_t^{obs}-\hat y_t$. The first four detectors check $|e_t|>\theta$; each $\theta$ is the 99th percentile of absolute residuals on unmodified validation.

In [ ]:

def predict_ols(model,observed):
    """Rebuild observed inputs and apply frozen OLS coefficients in ppm."""
    available=ELIGIBLE_IDX.intersection(observed.index)
    return predict_ols_features(model,make_features(observed).loc[available])

def residual_score(observed,predictor):
    """Return signed residuals: observed CO2 minus predicted CO2, in ppm."""
    prediction=predictor(observed)
    return observed.loc[prediction.index,"Carbon dioxide"]-prediction

fit_seconds = {}
calibration_seconds = {}
methods={}
model_metadata={}
def register_method(key,label,predictor,score_function,score_label):
    # Calibrate only on original validation; the prefix has no test timestamps.
    """Calibrate a point threshold on unmodified validation and register the detector; raise ValueError for non-finite scores."""
    _calibration_started = perf_counter()
    validation_score=score_function(calibration_observations).loc[VALIDATION_IDX]
    if not np.isfinite(validation_score.to_numpy()).all():
        raise ValueError('Validation scores contain missing or nonfinite values.')
    threshold=float(validation_score.abs().quantile(THRESHOLD_QUANTILE))
    methods[key]={"label":label,"predictor":predictor,"score_function":score_function,
                  "threshold":threshold,"score_label":score_label,
                  "threshold_calibration":"validation absolute-score quantile",
                  "validation_alert_rate":float((validation_score.abs()>threshold).mean())}
    calibration_seconds[key] = perf_counter() - _calibration_started
    print(label,': validation threshold',round(threshold,3),'ppm; fault labels are not used for calibration.')

print('Helpers ready; method construction starts in the next cell.')


**Code cell 8**

** Method 1: adjacent change rule**

Use the previous observation as a persistence prediction. No model fit is needed; validation still sets the change threshold.

In [ ]:
def predict_persistence(observed):
    """Return the previous observed CO2 in ppm as the change-rule reference."""
    available=ELIGIBLE_IDX.intersection(observed.index)
    return observed["Carbon dioxide"].shift(1).loc[available]

def score_change_rule(observed):
    """Return the current-minus-previous observed CO2 change in ppm."""
    return residual_score(observed,predict_persistence)

register_method("change_rule",'Adjacent change rule',predict_persistence,score_change_rule,'Adjacent change')
fit_seconds["change_rule"] = 0.0


**Code cell 9**

** Method 2: linear regression**

Fit OLS on original training rows. Rebuild test lags from corrupted observations. Record training and calibration times.

In [ ]:
_fit_started = perf_counter()
# Learn coefficients from original training data, not fault copies.
history_ols=fit_ols(original_features.loc[TRAIN_IDX,HISTORY_FEATURES],train_y)

fit_seconds["history_regression"] = perf_counter() - _fit_started

def predict_history_regression(observed):
    """Return history-regression CO2 predictions in ppm, using corrupted observed lags at test time."""
    return predict_ols(history_ols,observed)

def score_history_regression(observed):
    """Return signed history-regression residuals in ppm."""
    return residual_score(observed,predict_history_regression)

model_metadata["history_regression"]={"features":HISTORY_FEATURES,
    "standardized_coefficients":history_ols["coefficients"].tolist(),"fit_on":"original training only"}
register_method("history_regression",'History CO2 regression',predict_history_regression,score_history_regression,'Regression residual')


**Code cell 10**

** Method 3: neural network**

Fit the small 8-16-8-1 network. Fit input/target scaling on training only; no fault-label training.

In [ ]:

# Fit once on original training rows; scalers never use validation or test.
nn_fit=fit_neural(original_features.loc[TRAIN_IDX,HISTORY_FEATURES],train_y,MODEL_SEED)
neural_network=nn_fit.model
nn_x_mean,nn_x_scale=nn_fit.x_mean,nn_fit.x_scale
nn_y_mean,nn_y_scale=nn_fit.y_mean,nn_fit.y_scale
fit_warnings=nn_fit.warnings
fit_seconds['history_neural']=nn_fit.seconds
def predict_history_neural(observed):
    """Predict from observed inputs using the frozen NN; no refitting."""
    available=ELIGIBLE_IDX.intersection(observed.index)
    return nn_fit.predict(make_features(observed).loc[available])
def score_history_neural(observed):
    """Return observed-minus-predicted CO2 residuals in ppm."""
    return residual_score(observed,predict_history_neural)
model_metadata['history_neural']=nn_fit.metadata()
print('NN fitted; epochs:',neural_network.n_iter_)
register_method('history_neural','History CO2 neural network',predict_history_neural,score_history_neural,'NN residual')


**Code cell 11**

** Method 4: decision tree**

Use the same eight features, depth 5 and at least 50 rows per leaf. Compare model choice while retaining inputs.

In [ ]:
_fit_started = perf_counter()
history_tree=DecisionTreeRegressor(max_depth=TREE_MAX_DEPTH,
    min_samples_leaf=TREE_MIN_SAMPLES_LEAF,random_state=MODEL_SEED)
# Explicit training call; fault copies and fault labels are excluded.
history_tree.fit(original_features.loc[TRAIN_IDX,HISTORY_FEATURES],train_y)

fit_seconds["history_tree"] = perf_counter() - _fit_started

def predict_history_tree(observed):
    """Apply the fitted tree to the same eight observed features and return CO2 ppm predictions."""
    available=ELIGIBLE_IDX.intersection(observed.index)
    x=make_features(observed).loc[available,HISTORY_FEATURES]
    return pd.Series(history_tree.predict(x),index=available)

def score_history_tree(observed):
    """Return signed decision-tree prediction residuals in ppm."""
    return residual_score(observed,predict_history_tree)

model_metadata["history_tree"]={"features":HISTORY_FEATURES,"settings":history_tree.get_params(),
    "depth":int(history_tree.get_depth()),"leaves":int(history_tree.get_n_leaves()),
    "fit_on":"original training only"}
register_method("history_tree",'History CO2 decision tree',predict_history_tree,score_history_tree,'Tree regression residual')


**Code cell 12**

** CUSUM accumulation and calibration helpers**

Accumulate positive and negative residual shifts. Reset after alarms or large gaps. Training residuals set k; validation alone chooses h.

Center residuals as $r_t=e_t-\mu_{train}$, then accumulate:

$$S_t^+=\max(0,S_{t-1}^++r_t-k),\qquad S_t^-=\max(0,S_{t-1}^--r_t-k).$$

$$\text{score}_t=\max(S_t^+,S_t^-),\qquad \text{alarm}_t=(\text{score}_t>h).$$

Use the pre-reset score for the current alarm, then reset both accumulators. Set $k=0.1\sigma_{train}$ and search $h$ using validation only, with at most 25 validation alerts by default. Large gaps also reset state; known fault locations do not.

In [ ]:

def build_cusum_method(key,label,predictor,base_score_function):
    """Set center/k from training residuals and choose h within a validation alert budget; reuse the predictor without refitting."""
    _calibration_started = perf_counter()
    training_residual=base_score_function(original.loc[:TRAIN_IDX[-1]]).loc[TRAIN_IDX]
    center=float(training_residual.mean())
    sigma=float(training_residual.std(ddof=0))
    if not sigma>0 or not 0<CUSUM_K_FRACTION:
        raise ValueError('CUSUM requires varying training residuals and a positive k fraction.')
    k=CUSUM_K_FRACTION*sigma
    validation_residual=base_score_function(calibration_observations).loc[VALIDATION_IDX]-center
    # Match the validation alert budget: at most ceil(1%*2458)=25 by default.
    budget=int(np.ceil((1-THRESHOLD_QUANTILE)*len(VALIDATION_IDX)))
    choice=select_cusum_limit(validation_residual,k,sigma,THRESHOLD_QUANTILE,CUSUM_H_GRID)
    h,validation_alerts=choice['h'],choice['validation_alerts']
    trials=choice['trials']

    def score_function(observed):
        """Accumulate from validation into test without label-based resets and return the CUSUM statistic."""
        residual=base_score_function(observed)
        monitored=MONITORED_IDX.intersection(residual.index)
        # Carry validation state into test; never reset from event positions.
        return cusum_series(residual.loc[monitored]-center,k,h)

    methods[key]={"label":label,"predictor":predictor,"score_function":score_function,
        "threshold":h,"score_label":"CUSUM statistic",
        "threshold_calibration":"validation alert-budget search",
        "validation_alert_rate":validation_alerts/len(VALIDATION_IDX)}
    model_metadata[key]={"training_residual_center_ppm":center,"training_residual_sd_ppm":sigma,
        "k_fraction":CUSUM_K_FRACTION,"k":k,"h":h,"validation_alert_budget":budget,
        "validation_alerts":validation_alerts,"threshold_trials":trials,
        "reset_after_alarm":True,"fit_on":"training residuals and original validation only"}
    calibration_seconds[key] = perf_counter() - _calibration_started
    print(label,": k =",round(k,3),"；h =",round(h,3),'; validation alerts =',validation_alerts)

print('CUSUM helpers ready; existing predictions are reused without retraining predictors.')


**Code cell 13**

** Method 5: NN-CUSUM**

Reuse the NN from cell 13 without retraining. Change the alarm statistic, not the prediction; there is no fifth prediction curve.

In [ ]:
build_cusum_method("history_neural_cusum",'History NN residual CUSUM',
                   predict_history_neural,score_history_neural)

fit_seconds["history_neural_cusum"] = 0.0


**Code cell 14**

**Scale NN residuals by the reference threshold.** $z_t=(e_t-\mu_{train})/\theta_{NN}$. The reference is the original-validation 99th percentile of absolute NN residuals. A standard two-sided CUSUM uses allowance $\kappa$, threshold $H$, and resets after alarms. First verify exact scaling-only equivalence.

In [ ]:

NN_THETA = methods["history_neural"]["threshold"]
NN_CENTER = model_metadata["history_neural_cusum"]["training_residual_center_ppm"]
if not NN_THETA > 0:
    raise ValueError("The NN residual reference threshold must be positive.")
validation_residual = score_history_neural(calibration_observations).loc[VALIDATION_IDX]
validation_z = (validation_residual - NN_CENTER)/NN_THETA
old_meta = model_metadata["history_neural_cusum"]
old_scaled_kappa = old_meta["k"]/NN_THETA
old_scaled_H = old_meta["h"]/NN_THETA
# A change of units alone must not change the old detector's results.
scaled_control = normalized_cusum(validation_z, old_scaled_kappa, old_scaled_H)
old_control = cusum_series(validation_residual-NN_CENTER, old_meta["k"], old_meta["h"])
np.testing.assert_allclose(scaled_control.score*NN_THETA, old_control, atol=1e-8, rtol=1e-10)
np.testing.assert_array_equal(scaled_control.alarm, old_control > old_meta["h"])
print("NN reference threshold:", round(NN_THETA, 4), "ppm")
print("Original CUSUM in NN-threshold units: kappa =", round(old_scaled_kappa, 5), "; H =", round(old_scaled_H, 5))
print("Scaling-only control passed: changing units did not change alarms.")


**Code cell 15**

**Development only.** Simulate all three faults in validation copies at three seeds and magnitudes; long faults have three durations. Original validation stays unmodified and limits each candidate to about 1% alarms. Select the smallest feasible H for each kappa.

In [ ]:
new_calibration_start = perf_counter()

def development_cases():
    """Generate validation-copy faults only, then cache NN residuals before scoring labels.

    No test observations are passed to NN inference in this development stage.
    The original validation frame remains unchanged and calibrates alarm budgets.
    """
    cases = []
    for kind in ("spike", "drift", "bias"):
        for seed in DEV_SEEDS:
            for magnitude in DEV_MAGNITUDES:
                for duration in ((1,) if kind == "spike" else DEV_DURATIONS):
                    count = DEV_SPIKE_COUNT if kind == "spike" else DEV_LONG_COUNT
                    events = plan_windows(duration, count, seed, partition="validation")
                    obs = calibration_observations.copy()
                    offsets = pd.Series(0.0, index=obs.index)
                    for event in events:
                        profile = np.arange(1, duration+1)/duration if kind == "drift" else np.ones(duration)
                        offsets.loc[event["active"]] = event["sign"] * magnitude * profile
                    obs["Carbon dioxide"] += offsets
                    assert obs.index.max() == VALIDATION_IDX[-1]
                    assert (offsets.loc[split.reindex(obs.index) != "validation"] == 0).all()
                    pd.testing.assert_frame_equal(obs.loc[TRAIN_IDX], original.loc[TRAIN_IDX])
                    pd.testing.assert_frame_equal(obs.drop(columns="Carbon dioxide"), calibration_observations.drop(columns="Carbon dioxide"))
                    # Detector inputs stop at the end of validation; labels are separate.
                    p = predict_history_neural(obs).loc[VALIDATION_IDX]
                    z = (obs.loc[VALIDATION_IDX, "Carbon dioxide"]-p-NN_CENTER)/NN_THETA
                    cases.append({"kind": kind, "seed": seed, "magnitude_ppm": magnitude,
                        "duration": duration, "events": events, "z": z})
    return cases


development = development_cases()
calibration_rows, development_rows = [], []
VALIDATION_BUDGET = int(np.ceil((1-THRESHOLD_QUANTILE)*len(VALIDATION_IDX)))
kappa_grid = sorted(set(KAPPA_GRID + [float(old_scaled_kappa)]))
for kappa in kappa_grid:
    chosen = None
    for H in NORMALIZED_H_GRID:
        trace = normalized_cusum(validation_z, kappa, float(H))
        alerts = int(trace.alarm.sum())
        calibration_rows.append({"kappa": kappa, "H": float(H), "validation_alerts": alerts,
                                 "validation_budget": VALIDATION_BUDGET})
        if alerts <= VALIDATION_BUDGET:
            chosen = (float(H), trace.alarm)
            break
    if chosen is None:
        continue
    H, clean_alarm = chosen
    for case in development:
        alarm = normalized_cusum(case["z"], kappa, H).alarm
        metrics = score_development_events(alarm, clean_alarm, case["events"])
        development_rows.append({"kappa": kappa, "H": H, "scenario": case["kind"],
            "seed": case["seed"], "magnitude_ppm": case["magnitude_ppm"], "duration_readings": case["duration"],
            "events": len(case["events"]), "validation_alerts": int(clean_alarm.sum()), **metrics})
if not development_rows:
    raise ValueError("No candidate satisfied the validation alarm budget.")
calibration_trials = pd.DataFrame(calibration_rows)
development_results = pd.DataFrame(development_rows)
calibration_trials.to_csv(RUN_DIR/"candidate_calibration_trials.csv", index=False)
development_results.to_csv(RUN_DIR/"development_candidate_results.csv", index=False)
manifest_rows = [{"scenario": c["kind"], "seed": c["seed"], "magnitude_ppm": c["magnitude_ppm"],
    "duration_readings": c["duration"], "events": [{"event_id": e["event_id"], "sign": e["sign"],
    "active": [t.isoformat() for t in e["active"]]} for e in c["events"]]} for c in development]
(RUN_DIR/"development_manifest.json").write_text(json.dumps(manifest_rows, indent=2), encoding="utf-8")
print("Development scenarios:", len(development), "; validation alarm budget:", VALIDATION_BUDGET)


**Code cell 16**

**Select and freeze.** Maximize equal-type macro paired additional event recall; ties prefer smaller capped delay then fewer validation alarms. No test fault results exist yet. NN is reused and one CUSUM rule makes the new alarms.

In [ ]:
NEW_LABEL = 'NN threshold-scaled CUSUM'

# Give each fault TYPE equal weight, despite different numbers of duration cases.
development_by_kind = development_results.groupby(["kappa", "H", "scenario"], as_index=False).agg(
    additional_event_recall=("additional_event_recall", "mean"),
    event_recall=("event_recall", "mean"),
    capped_delay=("capped_additional_delay_fraction", "mean"),
    validation_alerts=("validation_alerts", "first"))
ranking = development_by_kind.groupby(["kappa", "H"], as_index=False).agg(
    macro_additional_recall=("additional_event_recall", "mean"),
    macro_event_recall=("event_recall", "mean"), macro_capped_delay=("capped_delay", "mean"),
    validation_alerts=("validation_alerts", "first"))
# Predeclared choice: greater macro paired-added recall; ties prefer less delay,
# then fewer unmodified-validation alarms, then larger H, then smaller kappa.
ranking = ranking.sort_values(["macro_additional_recall", "macro_capped_delay", "validation_alerts", "H", "kappa"],
                              ascending=[False, True, True, False, True]).reset_index(drop=True)
selected = ranking.iloc[0]
SELECTED_KAPPA, SELECTED_H = float(selected.kappa), float(selected.H)
NEW_KEY = "nn_threshold_cusum"

def score_nn_threshold_cusum(observations):
    """Use the same frozen NN, centered residual and selected single CUSUM rule."""
    residual = score_history_neural(observations)
    monitored = MONITORED_IDX.intersection(residual.index)
    z = (residual.loc[monitored]-NN_CENTER)/NN_THETA
    return normalized_cusum(z, SELECTED_KAPPA, SELECTED_H).score

methods[NEW_KEY] = {"label": NEW_LABEL, "predictor": predict_history_neural,
    "score_function": score_nn_threshold_cusum, "threshold": SELECTED_H,
    "score_label": "CUSUM / NN threshold", "threshold_calibration": "validation budget and development simulations",
    "validation_alert_rate": float(selected.validation_alerts/len(VALIDATION_IDX))}
fit_seconds[NEW_KEY] = 0.0  # Reuses the existing trained NN, not a second predictor.
calibration_seconds[NEW_KEY] = perf_counter()-new_calibration_start
model_metadata[NEW_KEY] = {"theta_nn_ppm": NN_THETA, "center_ppm": NN_CENTER,
    "kappa": SELECTED_KAPPA, "H": SELECTED_H, "equivalent_k_ppm": SELECTED_KAPPA*NN_THETA,
    "equivalent_h_ppm": SELECTED_H*NN_THETA, "selection_source": "faulted validation copies only",
    "macro_development_additional_recall": float(selected.macro_additional_recall),
    "validation_alerts": int(selected.validation_alerts), "validation_budget": VALIDATION_BUDGET,
    "single_cusum_alarm_rule": True, "nn_alarm_gate": False, "or_with_nn_point_alarm": False}
ranking.to_csv(RUN_DIR/"development_ranking.csv", index=False)
development_by_kind.to_csv(RUN_DIR/"development_by_kind.csv", index=False)
(RUN_DIR/"frozen_selection.json").write_text(json.dumps(model_metadata[NEW_KEY], indent=2), encoding="utf-8")
display(ranking.round(4))
display(development_by_kind[(development_by_kind.kappa == SELECTED_KAPPA) & (development_by_kind.H == SELECTED_H)].round(4))
print("Frozen candidate: kappa =", SELECTED_KAPPA, "; H =", SELECTED_H)
print("Equivalent allowance/threshold in ppm-scaled units:", round(SELECTED_KAPPA*NN_THETA, 3), round(SELECTED_H*NN_THETA, 3))
print("Selection is complete. No test fault results have been evaluated.")


**Code cell 17**

** Create spike faults**

Modify one CO2 reading per event. Observed = original record + simulated offset. Save labels separately.

In [ ]:
# Create spike on a separate copy. Prediction receives observations only.
spike_events=plan_windows(duration=1,count=SPIKE_COUNT,seed=FAULT_SEED)
spike_observed,spike_offsets=inject_fault(original,spike_events,'spike',SPIKE_PPM)
scenarios['spike']=save_scenario('spike',spike_observed,spike_offsets,spike_events)
display(spike_offsets.loc[spike_events[0]['active']].head().rename('injected_offset_ppm'))


**Code cell 18**

** Create drift faults**

Grow the offset linearly to the specified endpoint, then return to the original record. This does not represent every permanent drift.

In [ ]:
# Create drift on a separate copy. Prediction receives observations only.
drift_events=plan_windows(duration=DRIFT_READINGS,count=DRIFT_COUNT,seed=FAULT_SEED)
drift_observed,drift_offsets=inject_fault(original,drift_events,'drift',DRIFT_PPM)
scenarios['drift']=save_scenario('drift',drift_observed,drift_offsets,drift_events)
display(drift_offsets.loc[drift_events[0]['active']].head().rename('injected_offset_ppm'))


**Code cell 19**

** Create bias faults**

Add one constant signed offset throughout a window. The difference from the original is constant; the observed curve need not be horizontal.

In [ ]:
# Create bias on a separate copy. Prediction receives observations only.
bias_events=plan_windows(duration=BIAS_READINGS,count=BIAS_COUNT,seed=FAULT_SEED)
bias_observed,bias_offsets=inject_fault(original,bias_events,'bias',BIAS_PPM)
scenarios['bias']=save_scenario('bias',bias_observed,bias_offsets,bias_events)
display(bias_offsets.loc[bias_events[0]['active']].head().rename('injected_offset_ppm'))


**Code cell 20**

** Inspect five methods and thresholds**

Check that all five methods exist and inspect validation alert rates. Cumulative and instantaneous thresholds are not directly comparable.

In [ ]:
EXPECTED_METHODS = ["change_rule", "history_regression", "history_neural", "history_tree", "history_neural_cusum", NEW_KEY]
assert list(methods) == EXPECTED_METHODS
threshold_table = pd.DataFrame([{"method": key, "name": method["label"],
    "threshold": method["threshold"], "score_kind": method["score_label"],
    "threshold_calibration": method["threshold_calibration"],
    "validation_alert_rate": method["validation_alert_rate"]} for key, method in methods.items()])
display(threshold_table.round(4))
threshold_table.to_csv(RUN_DIR / "thresholds.csv", index=False)
print('Point thresholds use validation residual quantiles; CUSUM uses a validation alert budget. Their statistics and thresholds have different meanings.')


**Code cell 21**

**Final test comparison.** Frozen parameters, identical faults, four distinct predictions. One NN forward prediction is shared by its three alarm rules. Inference precedes label scoring.

In [ ]:
inference_timing_rows = []

def infer_scenario(observations, phase="unspecified"):
    """Observation-only six-detector replay; one NN forward prediction shared by three rules.

    The detector never receives original test reference values or fault labels.
    CUSUM replays validation then test; there are no event-boundary resets.
    """
    cache, prediction_cost, inferred = {}, {}, {}
    for key in ("change_rule", "history_regression", "history_neural", "history_tree"):
        started = perf_counter()
        cache[key] = methods[key]["predictor"](observations)
        prediction_cost[key] = perf_counter()-started
    for key, method in methods.items():
        shared = key in ("history_neural_cusum", NEW_KEY)
        full_prediction = cache["history_neural" if shared else key]
        residual = observations.loc[full_prediction.index, "Carbon dioxide"]-full_prediction
        started = perf_counter()
        monitored = MONITORED_IDX.intersection(residual.index)
        if key == "history_neural_cusum":
            meta = model_metadata[key]
            score = cusum_series(residual.loc[monitored]-meta["training_residual_center_ppm"], meta["k"], meta["h"])
        elif key == NEW_KEY:
            z = (residual.loc[monitored]-NN_CENTER)/NN_THETA
            score = normalized_cusum(z, SELECTED_KAPPA, SELECTED_H).score
        else:
            score = residual
        score = score.loc[TEST_IDX]
        statistic_seconds = perf_counter()-started
        prediction = full_prediction.loc[TEST_IDX]
        if not np.isfinite(prediction.to_numpy()).all() or not np.isfinite(score.to_numpy()).all():
            raise ValueError("Nonfinite predictions or test statistics.")
        inferred[key] = pd.DataFrame({"prediction_ppm": prediction, "residual_ppm": residual.loc[TEST_IDX],
            "score": score, "alarm": score.abs() > method["threshold"]}, index=TEST_IDX)
        prediction_seconds = 0.0 if shared else prediction_cost[key]
        inference_timing_rows.append({"phase": phase, "method": key, "input_rows": len(observations),
            "prediction_rows": len(full_prediction), "test_rows": len(TEST_IDX),
            "prediction_seconds": prediction_seconds, "statistic_seconds": statistic_seconds,
            "total_seconds": prediction_seconds+statistic_seconds, "reused_nn_prediction": shared})
    return inferred

inferred_by_scenario={name:infer_scenario(scenario["observations"], phase=name) for name,scenario in scenarios.items()}
unmodified_inference=infer_scenario(original, phase="test_unmodified")  # Evaluation baseline, not an input to fault-copy prediction

summary_rows,event_rows=[],[]
for name,scenario in scenarios.items():
    reference=scenario['reference'].loc[TEST_IDX]
    result_folder=RUN_DIR/'results'/name
    result_folder.mkdir(parents=True,exist_ok=False)
    for key,points in inferred_by_scenario[name].items():
        metrics,details=evaluate_alarm_events(points.alarm,unmodified_inference[key].alarm,scenario['events'])
        summary_rows.append({'scenario':name,'method':key,'name':methods[key]['label'],**metrics})
        for row in details:
            event_rows.append({'scenario':name,'method':key,**{k:row[k] for k in ['event_id','detected','additional_alarm','first_alert_delay_minutes']}})
        exported=points.copy()
        exported.insert(0,'observed_co2_ppm',scenario['observations'].loc[TEST_IDX,'Carbon dioxide'])
        exported['original_co2_ppm']=reference.original_co2_ppm
        exported['event_id']=reference.event_id
        exported['threshold']=methods[key]['threshold']
        exported.to_csv(result_folder/f'{key}_points.csv')
comparison=pd.DataFrame(summary_rows)
event_comparison=pd.DataFrame(event_rows)
comparison.to_csv(RUN_DIR/"comparison_all_methods.csv",index=False)
event_comparison.to_csv(RUN_DIR/"event_results.csv",index=False)
# Show the main columns first; the CSV stores all metrics.
display(comparison[["scenario","name","detected","events","paired_additional_events",
                    "unmodified_test_alert_rate"]].round(4))
print('detected: events with an alarm during injection; paired_additional_events: events with an added alarm relative to the unmodified copy at the same timestamps.')
print('Unmodified alert rates describe background alerts, not verified false-positive rates; original health is unknown.')
print('Paired additional event: at least one active timestamp alarms after injection but not at that same timestamp in the unmodified replay. The entire unmodified window need not be alarm-free.')


**Code cell 22**

** Prediction errors, matched delays and timings**

Use MAE/RMSE as supporting evidence. Pair NN/CUSUM delays on commonly detected events and report timing separately.

In [ ]:
PREDICTION_METHODS=["change_rule","history_regression","history_neural","history_tree"]
prediction_rows=[]
for key in PREDICTION_METHODS:
    original_prediction=methods[key]["predictor"](original)
    for part,index in (("train",TRAIN_IDX),("validation",VALIDATION_IDX),("test_unmodified",TEST_IDX)):
        error=original.loc[index,"Carbon dioxide"]-original_prediction.loc[index]
        prediction_rows.append({"method":key,"name":methods[key]["label"],"part":part,
                                "mae_ppm":float(error.abs().mean()),"rmse_ppm":float(np.sqrt(np.mean(error**2)))})
prediction_quality=pd.DataFrame(prediction_rows)
prediction_quality.to_csv(RUN_DIR/"prediction_quality.csv",index=False)
display(prediction_quality.pivot(index="name",columns="part",values="mae_ppm").round(2))
print('The table reports prediction MAE; accurate prediction does not guarantee drift detection.')

injected_prediction_rows=[]
for name,scenario in scenarios.items():
    active=TEST_IDX[scenario["reference"].loc[TEST_IDX,"event_id"]>0]
    baseline=scenario["reference"].loc[active,"original_co2_ppm"]
    for key in PREDICTION_METHODS:
        prediction=inferred_by_scenario[name][key].loc[active,"prediction_ppm"]
        error=prediction-baseline
        injected_prediction_rows.append({"scenario":name,"method":key,
            "prediction_vs_original_injected_mae_ppm":float(error.abs().mean()),
            "prediction_vs_original_injected_rmse_ppm":float(np.sqrt(np.mean(error**2)))})
injected_prediction_quality=pd.DataFrame(injected_prediction_rows)
injected_prediction_quality.to_csv(RUN_DIR/"prediction_vs_original_injected.csv",index=False)
print('Prediction errors against the pre-injection record are saved in prediction_vs_original_injected.csv.')

# Pair the SAME scenario/event rather than comparing different detected subsets.
pair_keys = ["scenario", "event_id"]
nn_events = event_comparison.query('method == "history_neural"')[pair_keys + ["detected", "first_alert_delay_minutes"]]
cs_events = event_comparison.query('method == "history_neural_cusum"')[pair_keys + ["detected", "first_alert_delay_minutes"]]
paired_delay_events = nn_events.merge(cs_events, on=pair_keys, suffixes=("_nn", "_cusum"), validate="one_to_one")
paired_delay_events["both_detected"] = paired_delay_events.detected_nn & paired_delay_events.detected_cusum
paired_delay_events["cusum_minus_nn_minutes"] = (
    paired_delay_events.first_alert_delay_minutes_cusum - paired_delay_events.first_alert_delay_minutes_nn
).where(paired_delay_events.both_detected)
matched_rows = []
for name, group in paired_delay_events.groupby("scenario", sort=False):
    common = group[group.both_detected]
    matched_rows.append({"scenario": name, "events": len(group), "both_detected": len(common),
        "nn_only": int((group.detected_nn & ~group.detected_cusum).sum()),
        "cusum_only": int((~group.detected_nn & group.detected_cusum).sum()),
        "neither": int((~group.detected_nn & ~group.detected_cusum).sum()),
        "matched_nn_median_minutes": float(common.first_alert_delay_minutes_nn.median()) if len(common) else None,
        "matched_cusum_median_minutes": float(common.first_alert_delay_minutes_cusum.median()) if len(common) else None,
        "median_paired_cusum_minus_nn_minutes": float(common.cusum_minus_nn_minutes.median()) if len(common) else None})
matched_delay_summary = pd.DataFrame(matched_rows)
paired_delay_events.to_csv(RUN_DIR / "nn_cusum_paired_event_delays.csv", index=False)
matched_delay_summary.to_csv(RUN_DIR / "nn_cusum_matched_delay_summary.csv", index=False)
display(matched_delay_summary.round(2))
print('Delays are paired only for events detected by both. CUSUM minus NN below zero means earlier CUSUM detection. Leave undefined delays blank; never replace missed detections with zero.')

inference_timing = pd.DataFrame(inference_timing_rows)
timing_rows = []
for key in methods:
    samples = inference_timing[inference_timing.method == key]
    timing_rows.append({"method": key, "name": methods[key]["label"],
        "training_seconds": fit_seconds.get(key, 0.0),
        "shared_trained_predictor": "history_neural" if key in ("history_neural_cusum", NEW_KEY) else "",
        "calibration_seconds": calibration_seconds[key], "replays": len(samples),
        "median_prediction_seconds": float(samples.prediction_seconds.median()),
        "median_statistic_seconds": float(samples.statistic_seconds.median()),
        "median_total_seconds": float(samples.total_seconds.median())})
timing_summary = pd.DataFrame(timing_rows)
inference_timing.to_csv(RUN_DIR / "inference_timing.csv", index=False)
timing_summary.to_csv(RUN_DIR / "timing_summary.csv", index=False)
display(timing_summary.round(4))
print('Timings measure this batch replay: prediction covers the full input and statistics cover the relevant monitored rows. File I/O, evaluation and plotting are excluded. These are not per-reading online latencies. CUSUM reuses NN training but still needs NN training and prediction.')

# Compare identical detected events for the old and new CUSUM, not unlike subsets.
left=event_comparison[event_comparison.method=="history_neural_cusum"]
right=event_comparison[event_comparison.method==NEW_KEY]
old_new_pairs=left.merge(right,on=["scenario","event_id"],suffixes=("_old","_new"),validate="one_to_one")
old_new_pairs["both_detected"]=old_new_pairs.detected_old & old_new_pairs.detected_new
old_new_pairs["new_minus_old_delay_minutes"]=(old_new_pairs.first_alert_delay_minutes_new-old_new_pairs.first_alert_delay_minutes_old).where(old_new_pairs.both_detected)
old_new_pairs.to_csv(RUN_DIR/"old_new_paired_delays.csv",index=False)
display(old_new_pairs[["scenario","event_id","detected_old","detected_new","new_minus_old_delay_minutes"]].round(2))


**Code cell 23**

** Compact local plots for three faults**

Default to event 1 of each fault with six context readings on either side. Show all four predictions; red crosses identify one selected detector. Rerun only this cell for view changes.

In [ ]:
PLOT_SCENARIOS=["spike","drift","bias"]  # Use ["drift"], for example, to show just one panel.
PLOT_EVENT_IDS={"spike":1,"drift":1,"bias":1}  # Select one event for each fault type.
ALARM_METHOD=NEW_KEY  # Controls red crosses only; all four distinct predictions remain visible.
POINTS_BEFORE=6
POINTS_AFTER=6
SHOW_ALARM_TABLE=False

if not PLOT_SCENARIOS or len(set(PLOT_SCENARIOS))!=len(PLOT_SCENARIOS):
    raise ValueError('Select at least one fault type, with no duplicates.')
if any(name not in scenarios for name in PLOT_SCENARIOS) or ALARM_METHOD not in methods:
    raise ValueError('Check the fault types or ALARM_METHOD name.')
if POINTS_BEFORE<0 or POINTS_AFTER<0:
    raise ValueError('Context point counts must be nonnegative.')

prediction_styles={
    "change_rule":("tab:brown",":","Previous reading (change rule)"),
    "history_regression":("tab:blue","--","Linear regression"),
    "history_neural":("tab:purple","-.","Neural network (also CUSUM reference)"),
    "history_tree":("tab:cyan",(0,(4,1)),"Decision tree"),
}
# Prepare local events from cell 18 results without training models.
panels=[]
for name in PLOT_SCENARIOS:
    scenario=scenarios[name]
    event_id=PLOT_EVENT_IDS.get(name,1)
    event=next((e for e in scenario["events"] if e["event_id"]==event_id),None)
    if event is None:
        raise ValueError(f"{name} has no event {event_id}; check PLOT_EVENT_IDS.")
    first,last=TEST_IDX.get_loc(event["active"][0]),TEST_IDX.get_loc(event["active"][-1])
    local=TEST_IDX[max(0,first-POINTS_BEFORE):min(len(TEST_IDX),last+POINTS_AFTER+1)]
    start=event["active"][0]
    minutes=(local-start).total_seconds()/60  # Zero is the first injected timestamp for this event.
    panels.append({"name":name,"event_id":event_id,"event":event,"scenario":scenario,
        "local":local,"minutes":minutes,
        "observed":scenario["observations"].loc[local,"Carbon dioxide"],
        "reference":scenario["reference"].loc[local,"original_co2_ppm"],
        "results":inferred_by_scenario[name]})

height=3.0*len(panels)+1.7
fig,axes_grid=plt.subplots(len(panels),1,figsize=(11,height),sharey=True,squeeze=False)
axes=axes_grid[:,0]
plot_prediction_rows,plot_alarm_rows,y_values=[],[],[]
for ax,panel in zip(axes,panels):
    name,event_id,event=panel["name"],panel["event_id"],panel["event"]
    local,minutes=panel["local"],panel["minutes"]
    observed,reference=panel["observed"],panel["reference"]
    results=panel["results"]
    # Gray observed = original plus simulated offset; black is an evaluation reference.
    ax.plot(minutes,reference,color="black",lw=2,label="Original record (pre-injection)")
    ax.plot(minutes,observed,color="0.65",lw=1.6,label="Observed (simulated fault)")
    y_values.extend([reference.to_numpy(),observed.to_numpy()])
    for key,(color,style,label) in prediction_styles.items():
        prediction=results[key].loc[local,"prediction_ppm"]
        ax.plot(minutes,prediction,color=color,ls=style,lw=1.6,label=label)
        y_values.append(prediction.to_numpy())
        error=prediction-reference
        active_error=results[key].loc[event["active"],"prediction_ppm"]-panel["scenario"]["reference"].loc[event["active"],"original_co2_ppm"]
        plot_prediction_rows.append({"scenario":name,"event_id":event_id,"method":key,
            "name":methods[key]["label"],"local_readings":len(local),
            "local_mae_vs_original_ppm":float(error.abs().mean()),
            "injected_mae_vs_original_ppm":float(active_error.abs().mean())})

    alarm_points=results[ALARM_METHOD].loc[local]
    hits=local[alarm_points.alarm.to_numpy()]
    hit_minutes=(hits-event["active"][0]).total_seconds()/60
    ax.scatter(hit_minutes,observed.loc[hits],color="red",marker="x",s=55,zorder=5,
               label=f"Alarm: {ALARM_METHOD}")
    for number,(timestamp,x_value) in enumerate(zip(hits,hit_minutes),1):
        ax.annotate(f"A{number}",(x_value,observed.loc[timestamp]),xytext=(6,6),
                    textcoords="offset points",color="red",fontsize=8)
        plot_alarm_rows.append({"scenario":name,"event_id":event_id,"alarm_id":f"A{number}",
            "minutes_from_start":float(x_value),"observed_ppm":float(observed.loc[timestamp]),
            "prediction_ppm":float(alarm_points.loc[timestamp,"prediction_ppm"]),
            "score":float(alarm_points.loc[timestamp,"score"]),
            "threshold":methods[ALARM_METHOD]["threshold"],
            "inside_injected_event":bool(panel["scenario"]["reference"].loc[timestamp,"event_id"]==event_id)})
    end_minutes=(event["active"][-1]-event["active"][0]).total_seconds()/60
    if len(event["active"])==1:
        ax.axvline(0,color="orange",lw=1.5,label="Injected event")
    else:
        ax.axvspan(0,end_minutes,color="orange",alpha=.13,label="Injected event")
    descriptions={"spike":"sudden single-reading error","drift":"gradually increasing offset",
                  "bias":"constant offset from original"}
    ax.set_title(f"{name.capitalize()}: {descriptions[name]} | Event {event_id}",loc="left",fontsize=11)
    ax.set_ylabel("CO2 (ppm)")
    ax.set_xlabel("Minutes from event start (0 = first injected reading)")
    ax.grid(alpha=.2)
    ax.margins(x=.04)

# Common CO2 limits support magnitude comparisons; time ranges fit each event.
all_y=np.concatenate(y_values)
lower,upper=float(np.min(all_y)),float(np.max(all_y))
padding=max(5.0,.08*(upper-lower))
axes[0].set_ylim(lower-padding,upper+padding)
# Use one shared legend rather than repeating it in every panel.
handles,labels=axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc="lower center",ncol=2,fontsize=8.5,bbox_to_anchor=(.5,.01))
fig.suptitle(f"CO2 fault examples | All predictions | Alarms: {ALARM_METHOD}",fontsize=12,y=.995)
fig.tight_layout(rect=(0,1.35/height,1,.96))
(RUN_DIR/"figures").mkdir(exist_ok=True)
panel_key="_".join(f"{p['name']}{p['event_id']}" for p in panels)
figure_path=RUN_DIR/"figures"/f"{panel_key}_all_predictions_alarm_{ALARM_METHOD}.png"
fig.savefig(figure_path,dpi=160,bbox_inches="tight")
plt.show()
plt.close(fig)

# Show selected-event injection MAE; use cells 18-19 for overall evaluation.

plot_prediction_comparison=pd.DataFrame(plot_prediction_rows)
display(plot_prediction_comparison.pivot(index="scenario",columns="name", values="injected_mae_vs_original_ppm").reindex(PLOT_SCENARIOS).round(2))
alarm_columns=["scenario","event_id","alarm_id","minutes_from_start","observed_ppm", "prediction_ppm","score","threshold","inside_injected_event"]
plot_alarm_table=pd.DataFrame(plot_alarm_rows,columns=alarm_columns)
if SHOW_ALARM_TABLE:
    display(plot_alarm_table.round(2))
print('Six detectors have four distinct predictions: both CUSUM rules share the NN prediction. Red crosses belong only to ALARM_METHOD. Local examples do not establish overall performance.')


**Code cell 24**

**Trace the new rule.** Top: shared NN prediction. Middle: scaled residual. Bottom: original and new scaled CUSUM scores and thresholds. Default examples are spike 1, drift 2, bias 2. Crosses belong to the new rule; outside shading they are recovery/context alarms.

In [ ]:
TRACE_EVENT_IDS = {"spike": 1, "drift": 2, "bias": 2}
TRACE_CONTEXT = 6
fig, axes = plt.subplots(3, 3, figsize=(15, 9), layout="constrained")
trace_tables = []
for col, kind in enumerate(("spike", "drift", "bias")):
    scenario = scenarios[kind]
    event = next(e for e in scenario["events"] if e["event_id"] == TRACE_EVENT_IDS[kind])
    active = event["active"]
    a, b = TEST_IDX.get_loc(active[0]), TEST_IDX.get_loc(active[-1])
    local = TEST_IDX[max(0,a-TRACE_CONTEXT):min(len(TEST_IDX),b+TRACE_CONTEXT+1)]
    minutes = np.asarray((local-active[0]).total_seconds()/60)
    points = inferred_by_scenario[kind]
    obs = scenario["observations"].loc[local,"Carbon dioxide"]
    axes[0,col].plot(minutes, scenario["reference"].loc[local,"original_co2_ppm"], color="black", label="Original record")
    axes[0,col].plot(minutes, obs, color="0.6", label="Observed + simulated fault")
    axes[0,col].plot(minutes, points["history_neural"].loc[local,"prediction_ppm"], color="#0072B2", ls="--", label="Shared NN prediction")
    axes[0,col].set_title(f"{kind.capitalize()} | Event {event['event_id']}")
    axes[0,col].set_ylabel("CO2 (ppm)")
    z = (points[NEW_KEY].loc[local,"residual_ppm"]-NN_CENTER)/NN_THETA
    axes[1,col].plot(minutes, z, color="#0072B2", label="Centered residual / NN threshold")
    axes[1,col].axhline(1, color="0.4", ls=":", label="Reference magnitude 1")
    axes[1,col].axhline(-1, color="0.4", ls=":")
    axes[1,col].set_ylabel("Scaled residual z")
    old_scaled_score = points["history_neural_cusum"].loc[local,"score"]/NN_THETA
    new_score = points[NEW_KEY].loc[local,"score"]
    axes[2,col].plot(minutes, old_scaled_score, color="#D55E00", ls="--", label="Original CUSUM / NN threshold")
    axes[2,col].plot(minutes, new_score, color="#009E73", label="Selected scaled CUSUM")
    axes[2,col].axhline(old_scaled_H, color="#D55E00", ls=":", label="Original scaled H")
    axes[2,col].axhline(SELECTED_H, color="#009E73", ls=":", label="Selected H")
    hits = points[NEW_KEY].loc[local,"alarm"].to_numpy()
    axes[2,col].scatter(minutes[hits], new_score.to_numpy()[hits], color="red", marker="x", s=40, label="New detector alarm", zorder=4)
    axes[2,col].set_ylabel("CUSUM in NN-threshold units")
    axes[2,col].set_xlabel("Minutes from injection start")
    for ax in axes[:,col]:
        if len(active)==1:
            ax.axvline(0, color="#E69F00", alpha=.7)
        else:
            ax.axvspan(0,(active[-1]-active[0]).total_seconds()/60,color="#E69F00",alpha=.12)
        ax.grid(alpha=.2)
    table = pd.DataFrame({"scenario":kind,"event_id":event["event_id"],"original_ppm":scenario["reference"].loc[local,"original_co2_ppm"],
        "observed_ppm":obs,"nn_prediction_ppm":points[NEW_KEY].loc[local,"prediction_ppm"],"z":z,
        "nn_point_alarm":points["history_neural"].loc[local,"alarm"],"old_score_scaled":old_scaled_score,
        "old_alarm":points["history_neural_cusum"].loc[local,"alarm"],"new_score":new_score,
        "new_alarm":points[NEW_KEY].loc[local,"alarm"],"inside_injection":local.isin(active)})
    trace_tables.append(table)
for row in range(3):
    axes[row,0].legend(fontsize=7,loc="best")
fig.suptitle("One NN prediction; one selected CUSUM alarm rule. Shading = injection; crosses outside are recovery/context alarms.")
fig.savefig(RUN_DIR/"figures"/"nn_threshold_cusum_trace.png",dpi=160)
plt.show()
plt.close(fig)
trace_readings = pd.concat(trace_tables)
trace_readings.to_csv(RUN_DIR/"trace_readings.csv")
display(trace_readings[trace_readings.inside_injection].groupby("scenario",sort=False).head(3).round(3))


**Code cell 25**

**Sensitivity to positions, amplitudes and durations.** Parameters remain frozen. Seeds 7,42,99 share faults across methods. Do not tune from this table; the test period was previously inspected.

In [ ]:

def score_test_case(observations,offsets,events,spec):
    """Infer without labels, then apply the shared evaluator to all detectors."""
    inferred=infer_scenario(observations,phase='sensitivity')
    rows,event_rows=[],[]
    for key,points in inferred.items():
        m,details=evaluate_alarm_events(points.alarm,unmodified_inference[key].alarm,events)
        rows.append({**spec,'method':key,'events':m['events'],'detected':m['detected'],
            'paired_additional_events':m['paired_additional_events'],'reading_recall':m['reading_recall'],
            'unmodified_test_alert_rate':m['unmodified_test_alert_rate'],
            'outside_alert_rate':m['outside_fault_and_recovery_alert_rate'],'recovery_alert_rate':m['recovery_alert_rate'],
            'median_delay_minutes':m['median_first_alert_delay_minutes']})
        for d in details:
            event_rows.append({**spec,'method':key,'event_id':d['event_id'],'sign':d['sign'],
                'detected':d['detected'],'additional_alarm':d['additional_alarm'],'delay_minutes':d['first_alert_delay_minutes']})
    return rows,event_rows

sensitivity_rows, sensitivity_event_rows = [],[]
for kind in ("spike","drift","bias"):
    for seed in TEST_SEEDS:
        for magnitude in TEST_MAGNITUDES:
            for duration in ((1,) if kind=="spike" else TEST_DURATIONS):
                events = plan_windows(duration, SPIKE_COUNT if kind=="spike" else DRIFT_COUNT if kind=="drift" else BIAS_COUNT, seed)
                offsets = pd.Series(0.0,index=original.index)
                for e in events:
                    profile = np.arange(1,duration+1)/duration if kind=="drift" else np.ones(duration)
                    offsets.loc[e["active"]] = e["sign"]*magnitude*profile
                observed = original.copy()
                observed["Carbon dioxide"] += offsets
                pd.testing.assert_frame_equal(observed.loc[split!="test"],original.loc[split!="test"])
                spec={"scenario":kind,"seed":seed,"magnitude_ppm":magnitude,"duration_readings":duration}
                rows,event_rows=score_test_case(observed,offsets,events,spec)
                sensitivity_rows.extend(rows)
                sensitivity_event_rows.extend(event_rows)
sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity_events = pd.DataFrame(sensitivity_event_rows)
sensitivity_aggregate = sensitivity.groupby(["scenario","method","magnitude_ppm","duration_readings"],as_index=False).agg(
    runs=("seed","size"),events=("events","sum"),detected=("detected","sum"),
    paired_additional_events=("paired_additional_events","sum"))
sensitivity.to_csv(RUN_DIR/"test_sensitivity.csv",index=False)
sensitivity_events.to_csv(RUN_DIR/"test_sensitivity_events.csv",index=False)
sensitivity_aggregate.to_csv(RUN_DIR/"test_sensitivity_aggregate.csv",index=False)
display(sensitivity_aggregate[(sensitivity_aggregate.magnitude_ppm==60)&
    ((sensitivity_aggregate.duration_readings==24)|(sensitivity_aggregate.scenario=="spike"))])
print("Repeated fault placements reuse the same test period; they are not independent real faults.")


**Code cell 26**

**Check and export.** Verify scaling equivalence, state resets, causality, shared predictions and calibration budget. Save results and limitations. Background alerts are not confirmed false positives.

In [ ]:
# Meaningful checks of state, causality and calibration, not only a syntax check.
toy_idx = pd.date_range("2020-01-01",periods=8,freq="10min",tz="UTC")
toy = normalized_cusum(pd.Series([1.9]+[0.0]*7,index=toy_idx),.1,1.0)
assert bool(toy.alarm.iloc[0]) and not toy.alarm.iloc[1:].any()
toy_slow = normalized_cusum(pd.Series([.25]*8,index=toy_idx),.1,1.0)
assert not toy_slow.alarm.iloc[:6].any() and bool(toy_slow.alarm.iloc[6])
gap_idx=pd.DatetimeIndex(["2020-01-01T00:00Z","2020-01-01T00:10Z","2020-01-01T01:00Z"])
gap_trace=normalized_cusum(pd.Series([.5,.5,.5],index=gap_idx),0.0,2.0)
np.testing.assert_allclose(gap_trace.score,[.5,1.0,.5])
for bad_k,bad_H in ((-1,1),(0,0),(np.nan,1)):
    try:
        normalized_cusum(pd.Series([0.0],index=toy_idx[:1]),bad_k,bad_H)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid parameters were accepted.")

for kind,scenario in scenarios.items():
    full = methods["history_neural"]["predictor"](scenario["observations"])
    err = scenario["observations"].loc[full.index,"Carbon dioxide"]-full
    idx=MONITORED_IDX.intersection(err.index)
    control=normalized_cusum((err.loc[idx]-NN_CENTER)/NN_THETA,old_scaled_kappa,old_scaled_H)
    expected=inferred_by_scenario[kind]["history_neural_cusum"]
    np.testing.assert_array_equal(control.loc[TEST_IDX,"alarm"],expected.alarm)
    np.testing.assert_allclose(control.loc[TEST_IDX,"score"]*NN_THETA,expected.score,atol=1e-8,rtol=1e-10)
    np.testing.assert_array_equal(inferred_by_scenario[kind][NEW_KEY].alarm,
        inferred_by_scenario[kind][NEW_KEY].score>SELECTED_H)
    np.testing.assert_array_equal(inferred_by_scenario[kind][NEW_KEY].prediction_ppm,
        inferred_by_scenario[kind]["history_neural"].prediction_ppm)

pivot=TEST_IDX[len(TEST_IDX)//2]
changed=original.copy()
changed.loc[changed.index>pivot,"Carbon dioxide"]+=500
future_result=infer_scenario(changed,phase="causality_check")
for key in methods:
    pd.testing.assert_frame_equal(future_result[key].loc[:pivot],unmodified_inference[key].loc[:pivot])
assert int(normalized_cusum(validation_z,SELECTED_KAPPA,SELECTED_H).alarm.sum())<=VALIDATION_BUDGET
assert file_sha256(INPUT_CSV)==SOURCE_HASH
selection_file=json.loads((RUN_DIR/"frozen_selection.json").read_text(encoding="utf-8"))
assert selection_file["kappa"]==SELECTED_KAPPA and selection_file["H"]==SELECTED_H
assert neural_network.n_features_in_==8
checks={"same_nn_predictions_for_point_original_and_new_cusum":True,
    "scaling_only_control_preserves_old_alarms_validation_and_three_test_scenarios":True,
    "single_cusum_alarm_without_nn_gate_or_or_rule":True,"large_and_persistent_toy_examples":True,
    "alarm_reset_and_gap_reset":True,"invalid_parameters_rejected":True,
    "future_changes_do_not_change_prior_outputs_all_six":True,"validation_alarm_budget_met":True,
    "selection_frozen_before_test_inference":True,"development_inputs_exclude_test_timestamps":True,
    "test_faults_leave_training_validation_and_environment_unchanged":True,"source_csv_unchanged":True}
report={"study_version":STUDY_VERSION,"status":"exploratory on a previously inspected test period",
    "audit":audit,"split_counts":split.value_counts().to_dict(),"models":model_metadata,
    "thresholds":threshold_table.to_dict("records"),"plan":experiment_plan,"checks":checks,
    "selection":selection_file,"source_unchanged":True,
    "versions":{"python":sys.version.split()[0],"numpy":np.__version__,"pandas":pd.__version__,"sklearn":sklearn.__version__},
    "limitations":["Threshold scaling alone changes no detections; tuning may change results.",
        "Development uses faulted validation copies and is not an independent final evaluation.",
        "The test period was previously inspected; further results remain exploratory.",
        "Original readings are not confirmed healthy; background alerts are not verified false positives.",
        "Simulated finite faults do not establish real-fault performance.",
        "The same historical CO2 contamination remains in NN predictions."]}
(RUN_DIR/"run_report.json").write_text(json.dumps(report,ensure_ascii=False,indent=2,allow_nan=False),encoding="utf-8")
display(pd.DataFrame(list(checks.items()),columns=["check","passed"]))
print("Results saved to:",RUN_DIR)

record_run(PROJECT_DIR,'main',RUN_DIR)


Method reference: [NIST CUSUM control charts](https://www.itl.nist.gov/div898/handbook/pmc/section3/pmc323.htm). Scaling by a fixed NN threshold is a change of units; any performance change here comes from development-selected parameters.